# Domenska adaptacija Whisper Small modela za srpski govor iz oblasti poljoprivrede

Cilj eksperimenta je da se ispita uticaj domenske adaptacije modela `openai/whisper-small` na prepoznavanje srpskog govora iz oblasti poljoprivrede.

Korpus sadrži 800 audio-snimaka: 675 za obučavanje, 75 za validaciju i 50 za testiranje. Test skup pripada četvrtom govorniku koji nije korišćen tokom obučavanja. Glavne metrike su WER i CER na nivou korpusa. Originalni i adaptirani model testiraju se pod istim uslovima: FP32, `beam=1`, srpski jezik i isti test skup.

In [1]:
# Ова ћелија мора бити покренута пре import-а torch-а.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# num2words је потребан за идентичну нормализацију бројева у референци и хипотези.
!pip install -q jiwer num2words


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.8 MB/s eta 0:00:00


In [2]:
import gc
import re
import time
import unicodedata
import wave
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

import librosa
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from jiwer import wer, cer
from num2words import num2words
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

SEED = 42
MODEL_NAME = "openai/whisper-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Broj vidljivih GPU:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA: True
Broj vidljivih GPU: 1
GPU: Tesla T4


## 1. Učitavanje i provera korpusa

Transkripti su zadati u formatu `ID|tekst`. Naziv WAV fajla sadrži ID i broj govornika. Ovde se formiraju `train`, `validation` i `test` tabele i proverava da svih 800 ID-jeva postoji tačno jednom.

In [3]:
ROOT = Path("/kaggle/input/datasets/sajkobre/serbian-agriculture-whisper")
BASE = ROOT / "Fine_Tuning" / "Fine_Tuning"
TRAIN_DIR = BASE / "train"
VAL_DIR = BASE / "val"
TEST_DIR = BASE / "test"
TXT_FILE = ROOT / "sve_recenice.txt"

references = {}
with open(TXT_FILE, "r", encoding="utf-8-sig") as f:
    for line in f:
        line = line.strip()
        if line:
            number, text = line.split("|", 1)
            references[int(number)] = text.strip()

def extract_id(filename):
    match = re.match(r"(\d+)govornik\d+\.wav$", filename)
    if match is None:
        raise ValueError(f"Neispravan naziv WAV fajla: {filename}")
    return int(match.group(1))

def create_metadata(folder):
    rows = []
    for wav in folder.glob("*.wav"):
        idx = extract_id(wav.name)
        if idx not in references:
            raise ValueError(f"Nema referentnog teksta za {wav.name}")
        rows.append({"id": idx, "audio": str(wav), "text": references[idx]})
    return pd.DataFrame(rows).sort_values("id").reset_index(drop=True)

train_df = create_metadata(TRAIN_DIR)
val_df = create_metadata(VAL_DIR)
test_df = create_metadata(TEST_DIR)

assert len(references) == 800
assert len(train_df) == 675
assert len(val_df) == 75
assert len(test_df) == 50

all_ids = train_df["id"].tolist() + val_df["id"].tolist() + test_df["id"].tolist()
assert len(all_ids) == 800
assert len(set(all_ids)) == 800
assert set(all_ids) == set(range(1, 801))

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Ukupno:", len(all_ids))
print("✓ Svih 800 ID-jeva postoji tačno jednom.")
display(test_df.head())


Train: 675
Validation: 75
Test: 50
Ukupno: 800
✓ Svih 800 ID-jeva postoji tačno jednom.


,id,audio,text
0,751,/kaggle/input/datasets/sajkobre/serbian-agricu...,Kupac želi zimzelenu ogradu dugu približno tri...
1,752,/kaggle/input/datasets/sajkobre/serbian-agricu...,Za ovaj deo dvorišta potrebna nam je biljka ko...
2,753,/kaggle/input/datasets/sajkobre/serbian-agricu...,Pre sadnje tuje potrebno je pripremiti zemljiš...
3,754,/kaggle/input/datasets/sajkobre/serbian-agricu...,Kupac traži manje ukrasno drvo za prostor izme...
4,755,/kaggle/input/datasets/sajkobre/serbian-agricu...,Hortenzija posađena na jakom popodnevnom suncu...


In [4]:
def audio_statistics(df, name):
    durations, sample_rates, channels = [], [], []

    for path in df["audio"]:
        with wave.open(path, "rb") as w:
            sr = w.getframerate()
            durations.append(w.getnframes() / sr)
            sample_rates.append(sr)
            channels.append(w.getnchannels())

    result = {
        "skup": name,
        "broj": len(df),
        "trajanje_min": sum(durations) / 60,
        "prosek_s": np.mean(durations),
        "min_s": np.min(durations),
        "max_s": np.max(durations),
        "sample_rate": sorted(set(sample_rates)),
        "kanali": sorted(set(channels)),
    }
    return result, durations

train_stats, train_durations = audio_statistics(train_df, "Train")
val_stats, val_durations = audio_statistics(val_df, "Validation")
test_stats, test_durations = audio_statistics(test_df, "Test")

stats_df = pd.DataFrame([train_stats, val_stats, test_stats])
display(stats_df)

total_minutes = (sum(train_durations) + sum(val_durations) + sum(test_durations)) / 60
print(f"Ukupno trajanje: {total_minutes:.2f} min = {total_minutes/60:.2f} h")


,skup,broj,trajanje_min,prosek_s,min_s,max_s,sample_rate,kanali
0,Train,675,93.069130,8.272812,2.054966,21.083719,[44100],[1]
1,Validation,75,9.260311,7.408249,2.345215,12.538776,[44100],[1]
2,Test,50,9.225288,11.070346,9.380862,12.817415,[44100],[1]


Ukupno trajanje: 111.55 min = 1.86 h


### Rezultati provere korpusa

Potvrđeno je da korpus sadrži 800 snimaka, pravilno raspoređenih na skupove za obučavanje (675), validaciju (75) i testiranje (50). Ukupno trajanje audio materijala iznosi 111,55 minutaа.


In [5]:
def normalize_serbian_text(text):
    text = unicodedata.normalize("NFC", str(text)).lower()

    def number_to_words(match):
        return num2words(int(match.group(0)), lang="sr")

    text = re.sub(r"\b\d+\b", number_to_words, text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def evaluate_model(model, df, processor, label):
    model.eval()
    model.generation_config.language = "serbian"
    model.generation_config.task = "transcribe"
    model.generation_config.do_sample = False
    model.generation_config.num_beams = 1
    model.config.use_cache = True

    results = []

    for pos, (_, row) in enumerate(df.iterrows(), start=1):
        audio, sr = librosa.load(row["audio"], sr=16000, mono=True)
        duration = len(audio) / sr

        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(
            device=next(model.parameters()).device,
            dtype=next(model.parameters()).dtype
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()

        with torch.inference_mode():
            predicted_ids = model.generate(
                input_features,
                language="serbian",
                task="transcribe",
                num_beams=1,
                do_sample=False
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        processing_time = time.time() - start

        prediction = processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0].strip()

        ref_norm = normalize_serbian_text(row["text"])
        pred_norm = normalize_serbian_text(prediction)

        results.append({
            "id": row["id"],
            "audio": row["audio"],
            "reference": row["text"],
            "prediction": prediction,
            "reference_normalized": ref_norm,
            "prediction_normalized": pred_norm,
            "duration_s": duration,
            "processing_time_s": processing_time,
            "rtf": processing_time / duration,
            "wer": wer(ref_norm, pred_norm),
            "cer": cer(ref_norm, pred_norm),
        })

        print(f"[{pos:02d}/{len(df)}] ID={row['id']}")

    result_df = pd.DataFrame(results)

    metrics = {
        "model": label,
        "mean_wer": result_df["wer"].mean(),
        "median_wer": result_df["wer"].median(),
        "corpus_wer": wer(
            result_df["reference_normalized"].tolist(),
            result_df["prediction_normalized"].tolist()
        ),
        "mean_cer": result_df["cer"].mean(),
        "median_cer": result_df["cer"].median(),
        "corpus_cer": cer(
            result_df["reference_normalized"].tolist(),
            result_df["prediction_normalized"].tolist()
        ),
        "mean_rtf": result_df["rtf"].mean(),
    }
    return result_df, metrics

def print_metrics(m):
    print(f"\n{m['model']}")
    print(f"Prosečan WER:         {m['mean_wer']*100:.2f}%")
    print(f"Medijana WER-a:       {m['median_wer']*100:.2f}%")
    print(f"Corpus WER:           {m['corpus_wer']*100:.2f}%")
    print(f"Prosečan CER:         {m['mean_cer']*100:.2f}%")
    print(f"Medijana CER-a:       {m['median_cer']*100:.2f}%")
    print(f"Corpus CER:           {m['corpus_cer']*100:.2f}%")
    print(f"Prosečan RTF:         {m['mean_rtf']:.4f}")


## 3. Baseline: originalni Whisper Small

Baseline se meri pre obučavanja nad svih 50 test snimaka. Model je FP32 i koristi greedy dekodiranje (`beam=1`).

In [6]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

baseline_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32
).to(DEVICE)

baseline_df, baseline_metrics = evaluate_model(
    baseline_model, test_df, processor, "Original Whisper Small FP32"
)

print_metrics(baseline_metrics)

baseline_df.to_csv(
    "/kaggle/working/whisper_small_original_fp32_test50.csv",
    index=False,
    encoding="utf-8-sig"
)

baseline_model.cpu()
del baseline_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[01/50] ID=751
[02/50] ID=752
[03/50] ID=753
[04/50] ID=754
[05/50] ID=755
[06/50] ID=756
[07/50] ID=757
[08/50] ID=758
[09/50] ID=759
[10/50] ID=760
[11/50] ID=761
[12/50] ID=762
[13/50] ID=763
[14/50] ID=764
[15/50] ID=765
[16/50] ID=766
[17/50] ID=767
[18/50] ID=768
[19/50] ID=769
[20/50] ID=770
[21/50] ID=771
[22/50] ID=772
[23/50] ID=773
[24/50] ID=774
[25/50] ID=775
[26/50] ID=776
[27/50] ID=777
[28/50] ID=778
[29/50] ID=779
[30/50] ID=780
[31/50] ID=781
[32/50] ID=782
[33/50] ID=783
[34/50] ID=784
[35/50] ID=785
[36/50] ID=786
[37/50] ID=787
[38/50] ID=788
[39/50] ID=789
[40/50] ID=790
[41/50] ID=791
[42/50] ID=792
[43/50] ID=793
[44/50] ID=794
[45/50] ID=795
[46/50] ID=796
[47/50] ID=797
[48/50] ID=798
[49/50] ID=799
[50/50] ID=800

Original Whisper Small FP32
Prosečan WER:         31.30%
Medijana WER-a:       30.08%
Corpus WER:           31.01%
Prosečan CER:         5.91%
Medijana CER-a:       5.42%
Corpus CER:           5.91%
Prosečan RTF:         0.0741


## 4. Priprema podataka za fine-tuning

Whisper tokom obučavanja dobija originalni tekst transkripta. Audio se učitava kao mono signal i resampluje na 16 kHz. Proverava se i da nijedan target ne prelazi maksimalnu dužinu dekodera.

In [7]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[["id", "audio", "text"]], preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        val_df[["id", "audio", "text"]], preserve_index=False
    ),
})

MAX_TARGET_LENGTH = 448

train_lengths = np.array([
    len(processor.tokenizer(text, add_special_tokens=True).input_ids)
    for text in train_df["text"]
])
val_lengths = np.array([
    len(processor.tokenizer(text, add_special_tokens=True).input_ids)
    for text in val_df["text"]
])

assert (train_lengths < MAX_TARGET_LENGTH).all()
assert (val_lengths < MAX_TARGET_LENGTH).all()

print(dataset)
print("Maks. TRAIN tokena:", train_lengths.max())
print("Maks. VALIDATION tokena:", val_lengths.max())
print("✓ Svi transkripti staju u Whisper decoder limit.")


DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'text'],
        num_rows: 675
    })
    validation: Dataset({
        features: ['id', 'audio', 'text'],
        num_rows: 75
    })
})
Maks. TRAIN tokena: 108
Maks. VALIDATION tokena: 64
✓ Svi transkripti staju u Whisper decoder limit.


In [8]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = []

        for feature in features:
            audio, sr = librosa.load(feature["audio"], sr=16000, mono=True)
            mel = self.processor.feature_extractor(
                audio,
                sampling_rate=16000,
                return_tensors="pt"
            ).input_features[0]
            input_features.append({"input_features": mel})

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [
            {"input_ids": self.processor.tokenizer(feature["text"]).input_ids}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        if (
            labels.shape[1] > 0
            and (labels[:, 0] == self.processor.tokenizer.bos_token_id).all()
        ):
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = np.where(
        pred.label_ids == -100,
        processor.tokenizer.pad_token_id,
        pred.label_ids
    )

    pred_texts = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )
    label_texts = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    pred_norm = [normalize_serbian_text(x) for x in pred_texts]
    label_norm = [normalize_serbian_text(x) for x in label_texts]

    return {
        "wer": wer(label_norm, pred_norm),
        "cer": cer(label_norm, pred_norm),
    }

print("✓ Data collator i metrike su spremni.")


✓ Data collator i metrike su spremni.


## 5. Fine-tuning Whisper Small modela

Model se učitava u FP32. Tokom obučavanja koristi se AMP (`fp16=True`), dok FP32 parametri ostaju master parametri. Efektivna veličina batch-a je 8 (`2 × 4`). Obučavanje traje najviše 5 epoha, a najbolji checkpoint bira se prema najnižem validation WER-u.

In [9]:
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32
).to(DEVICE)

model.generation_config.language = "serbian"
model.generation_config.task = "transcribe"
model.generation_config.do_sample = False
model.generation_config.num_beams = 1
model.config.use_cache = False

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/whisper-small-serbian-agriculture",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=0.1,
    lr_scheduler_type="linear",
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    predict_with_generate=True,
    per_device_eval_batch_size=2,
    generation_max_length=448,
    generation_num_beams=1,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=25,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print("Trainer device:", trainer.args.device)
print("Broj GPU:", trainer.args.n_gpu)
print("Efektivni batch:", training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
print("Model dtype:", next(model.parameters()).dtype)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Trainer device: cuda:0
Broj GPU: 1
Efektivni batch: 8
Model dtype: torch.float32


In [10]:
train_result = trainer.train()

print("\nNajbolji checkpoint:", trainer.state.best_model_checkpoint)
print("Najbolji validation WER:", trainer.state.best_metric)


Epoch,Training Loss,Validation Loss,Wer,Cer
1,2.382798,0.551077,0.116558,0.017275
2,0.870208,0.120114,0.103486,0.014571
3,0.131149,0.100485,0.078431,0.011116
4,0.041596,0.099305,0.082789,0.011867
5,0.017757,0.101509,0.080610,0.011266


[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



Najbolji checkpoint: /kaggle/working/whisper-small-serbian-agriculture/checkpoint-255
Najbolji validation WER: 0.0784313725490196


In [11]:
# Sažetak validation rezultata po epohama
eval_rows = []

for item in trainer.state.log_history:
    if "eval_wer" in item:
        eval_rows.append({
            "epoch": item.get("epoch"),
            "validation_loss": item.get("eval_loss"),
            "validation_wer_pct": item["eval_wer"] * 100,
            "validation_cer_pct": item["eval_cer"] * 100,
        })

validation_history_df = pd.DataFrame(eval_rows)
display(validation_history_df)

best_row = validation_history_df.loc[
    validation_history_df["validation_wer_pct"].idxmin()
]
print(
    f"Najbolja epoha prema validation WER-u: "
    f"{best_row['epoch']:.0f} "
    f"(WER={best_row['validation_wer_pct']:.2f}%)"
)


,epoch,validation_loss,validation_wer_pct,validation_cer_pct
0,1.0,0.551077,11.655773,1.727505
1,2.0,0.120114,10.348584,1.457113
2,3.0,0.100485,7.843137,1.111612
3,4.0,0.099305,8.278867,1.186721
4,5.0,0.101509,8.061002,1.126634


Najbolja epoha prema validation WER-u: 3 (WER=7.84%)


### Rezultati obučavanja

Model je obučavan tokom pet epoha, uz efektivnu veličinu batch-a od 8. Najbolji rezultat na validacionom skupu ostvaren je nakon treće epohe, sa WER-om od 7,84%. Za završnu evaluaciju izabran je checkpoint sa najmanjom validacionom greškom.

In [12]:
FINAL_MODEL_DIR = "/kaggle/working/whisper-small-serbian-agriculture-best"

trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)

model = trainer.model
model.eval()
model.generation_config.language = "serbian"
model.generation_config.task = "transcribe"
model.generation_config.do_sample = False
model.generation_config.num_beams = 1
model.config.use_cache = True

print("Sačuvan model:", FINAL_MODEL_DIR)
print("Dtype:", next(model.parameters()).dtype)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sačuvan model: /kaggle/working/whisper-small-serbian-agriculture-best
Dtype: torch.float32


In [13]:
finetuned_df, finetuned_metrics = evaluate_model(
    model, test_df, processor, "Fine-tuned Whisper Small FP32"
)

print_metrics(finetuned_metrics)

finetuned_df.to_csv(
    "/kaggle/working/whisper_small_finetuned_test50.csv",
    index=False,
    encoding="utf-8-sig"
)


[01/50] ID=751
[02/50] ID=752
[03/50] ID=753
[04/50] ID=754
[05/50] ID=755
[06/50] ID=756
[07/50] ID=757
[08/50] ID=758
[09/50] ID=759
[10/50] ID=760
[11/50] ID=761
[12/50] ID=762
[13/50] ID=763
[14/50] ID=764
[15/50] ID=765
[16/50] ID=766
[17/50] ID=767
[18/50] ID=768
[19/50] ID=769
[20/50] ID=770
[21/50] ID=771
[22/50] ID=772
[23/50] ID=773
[24/50] ID=774
[25/50] ID=775
[26/50] ID=776
[27/50] ID=777
[28/50] ID=778
[29/50] ID=779
[30/50] ID=780
[31/50] ID=781
[32/50] ID=782
[33/50] ID=783
[34/50] ID=784
[35/50] ID=785
[36/50] ID=786
[37/50] ID=787
[38/50] ID=788
[39/50] ID=789
[40/50] ID=790
[41/50] ID=791
[42/50] ID=792
[43/50] ID=793
[44/50] ID=794
[45/50] ID=795
[46/50] ID=796
[47/50] ID=797
[48/50] ID=798
[49/50] ID=799
[50/50] ID=800

Fine-tuned Whisper Small FP32
Prosečan WER:         17.35%
Medijana WER-a:       16.67%
Corpus WER:           17.32%
Prosečan CER:         3.23%
Medijana CER-a:       2.97%
Corpus CER:           3.23%
Prosečan RTF:         0.1012


## 7. Konačno poređenje baseline i fine-tuned modela

Glavno poređenje koristi WER i CER na nivou korpusa. Dodatno se prikazuju apsolutno smanjenje u procentnim poenima i relativno smanjenje greške.

In [14]:
wer_reduction_pp = (
    baseline_metrics["corpus_wer"] - finetuned_metrics["corpus_wer"]
) * 100
wer_relative_reduction = (
    (baseline_metrics["corpus_wer"] - finetuned_metrics["corpus_wer"])
    / baseline_metrics["corpus_wer"]
) * 100

cer_reduction_pp = (
    baseline_metrics["corpus_cer"] - finetuned_metrics["corpus_cer"]
) * 100
cer_relative_reduction = (
    (baseline_metrics["corpus_cer"] - finetuned_metrics["corpus_cer"])
    / baseline_metrics["corpus_cer"]
) * 100

summary_df = pd.DataFrame([
    {
        "model": "Original Whisper Small FP32",
        "mean_WER_%": baseline_metrics["mean_wer"] * 100,
        "median_WER_%": baseline_metrics["median_wer"] * 100,
        "corpus_WER_%": baseline_metrics["corpus_wer"] * 100,
        "corpus_CER_%": baseline_metrics["corpus_cer"] * 100,
        "mean_RTF": baseline_metrics["mean_rtf"],
    },
    {
        "model": "Fine-tuned Whisper Small FP32",
        "mean_WER_%": finetuned_metrics["mean_wer"] * 100,
        "median_WER_%": finetuned_metrics["median_wer"] * 100,
        "corpus_WER_%": finetuned_metrics["corpus_wer"] * 100,
        "corpus_CER_%": finetuned_metrics["corpus_cer"] * 100,
        "mean_RTF": finetuned_metrics["mean_rtf"],
    },
])

display(summary_df.round(4))

print(f"WER: -{wer_reduction_pp:.2f} p.p. ({wer_relative_reduction:.2f}% relativno)")
print(f"CER: -{cer_reduction_pp:.2f} p.p. ({cer_relative_reduction:.2f}% relativno)")


,model,mean_WER_%,median_WER_%,corpus_WER_%,corpus_CER_%,mean_RTF
0,Original Whisper Small FP32,31.2965,30.0752,31.0063,5.9111,0.0741
1,Fine-tuned Whisper Small FP32,17.3517,16.6667,17.3164,3.2325,0.1012


WER: -13.69 p.p. (44.15% relativno)
CER: -2.68 p.p. (45.32% relativno)


## 8. Analiza po test snimku

Rezultati se spajaju po ID-ju kako bi se proverilo da li je poboljšanje raspoređeno na veći broj test primera, a ne samo na nekoliko izdvojenih snimaka.

In [15]:
comparison_df = baseline_df[
    ["id", "reference", "prediction", "wer", "cer"]
].merge(
    finetuned_df[
        ["id", "prediction", "wer", "cer"]
    ],
    on="id",
    how="inner",
    suffixes=("_baseline", "_finetuned")
)

assert len(comparison_df) == 50
assert comparison_df["id"].nunique() == 50

comparison_df["wer_change_pp"] = (
    comparison_df["wer_finetuned"] - comparison_df["wer_baseline"]
) * 100

comparison_df["cer_change_pp"] = (
    comparison_df["cer_finetuned"] - comparison_df["cer_baseline"]
) * 100

def classify_change(x, eps=1e-9):
    if x < -eps:
        return "poboljšan"
    if x > eps:
        return "pogoršan"
    return "isti"

comparison_df["status"] = comparison_df["wer_change_pp"].apply(classify_change)

improved = (comparison_df["status"] == "poboljšan").sum()
worsened = (comparison_df["status"] == "pogoršan").sum()
same = (comparison_df["status"] == "isti").sum()

print(f"Poboljšano: {improved}/50 ({improved/50*100:.1f}%)")
print(f"Pogoršano:  {worsened}/50 ({worsened/50*100:.1f}%)")
print(f"Bez promene: {same}/50 ({same/50*100:.1f}%)")

display(
    comparison_df[
        ["id", "wer_baseline", "wer_finetuned", "wer_change_pp", "status"]
    ].sort_values("wer_change_pp").head(10)
)

comparison_df.to_csv(
    "/kaggle/working/whisper_small_baseline_vs_finetuned_test50.csv",
    index=False,
    encoding="utf-8-sig"
)


Poboljšano: 45/50 (90.0%)
Pogoršano:  3/50 (6.0%)
Bez promene: 2/50 (4.0%)


,id,wer_baseline,wer_finetuned,wer_change_pp,status
26,777,0.578947,0.210526,-36.842105,poboljšan
9,760,0.363636,0.090909,-27.272727,poboljšan
16,767,0.550000,0.300000,-25.000000,poboljšan
38,789,0.500000,0.250000,-25.000000,poboljšan
29,780,0.380952,0.142857,-23.809524,poboljšan
15,766,0.428571,0.190476,-23.809524,poboljšan
42,793,0.409091,0.181818,-22.727273,poboljšan
24,775,0.363636,0.136364,-22.727273,poboljšan
10,761,0.434783,0.217391,-21.739130,poboljšan
37,788,0.315789,0.105263,-21.052632,poboljšan


### Rezultati po test snimcima

Adaptirani model ostvario je manji WER na 45 od 50 test snimaka (90%). Na tri snimka rezultat je pogoršan, dok je na dva ostao nepromenjen. Poboljšanje je, dakle, prisutno kod većine test primera.

In [16]:
from IPython.display import display, Markdown

# Pet najvećih poboljšanja prema smanjenju WER-a
top5 = (
    comparison_df
    .sort_values("wer_change_pp", ascending=True)
    .head(5)
    .reset_index(drop=True)
)

for i, row in top5.iterrows():

    wer_pre = row["wer_baseline"] * 100
    wer_posle = row["wer_finetuned"] * 100
    napredak = wer_pre - wer_posle

    display(Markdown(f"""
### Primer {i+1} — snimak ID {row['id']}

**Referentni tekst:**

{row['reference']}

**Originalni Whisper Small — WER: {wer_pre:.2f}%**

{row['prediction_baseline']}

**Fine-tuned Whisper Small — WER: {wer_posle:.2f}%**

{row['prediction_finetuned']}

**Smanjenje WER-a: {napredak:.2f} procentnih poena**

---
"""))

print("ZAVRŠNI REZULTAT DOMENSKE ADAPTACIJE")
print("=" * 45)

print(
    f"Originalni Whisper Small: "
    f"{baseline_metrics['corpus_wer']*100:.2f}% WER"
)

print(
    f"Fine-tuned Whisper Small: "
    f"{finetuned_metrics['corpus_wer']*100:.2f}% WER"
)

relativno = (
    1 - finetuned_metrics["corpus_wer"]
    / baseline_metrics["corpus_wer"]
) * 100

print(f"Relativno smanjenje WER-a: {relativno:.2f}%")


### Primer 1 — snimak ID 777

**Referentni tekst:**

Organska materija poboljšava strukturu zemljišta, doprinosi zadržavanju vode i predstavlja važan izvor hranljivih elemenata koji se postepeno oslobađaju mineralizacijom.

**Originalni Whisper Small — WER: 57.89%**

Organska materija pobešava struktur u zemljišta doprnosi za državanju vode i prestajua važan izbor hranjivih elementa koji se postepeno oslobađuju mineralizacijom.

**Fine-tuned Whisper Small — WER: 21.05%**

Organska materija pobešava strukturu zemljišta, doprnosi zadržavanju vode i predstavlja važan izbor hranljivih elemenata koji se postepeno oslobađuju mineralizacijom.

**Smanjenje WER-a: 36.84 procentnih poena**

---



### Primer 2 — snimak ID 760

**Referentni tekst:**

Prilikom preuzimanja većih sadnica potrebno je objasniti kupcu kako da ih pravilno transportuje, posadi, zalije i učvrsti tokom prvih nedelja nakon sadnje.

**Originalni Whisper Small — WER: 36.36%**

Prilekom preuzimanja većih sadnica potrebno objasniti kupcu kako da ih pravilno transporto je posad i zalije u čvrsti tokom prvih nedelja nakon sadnje.

**Fine-tuned Whisper Small — WER: 9.09%**

Prilikom preuzimanja većih sadnica potrebno objasniti kupcu kako da ih pravilno transportuje posadi zalije učvrsti tokom prvih nedelja nakon sadnje.

**Smanjenje WER-a: 27.27 procentnih poena**

---



### Primer 3 — snimak ID 767

**Referentni tekst:**

Uklanjanjem određenog broja listova oko grozdova može se poboljšati provetravanje, ali preterano izlaganje direktnom suncu može povećati rizik od ožegotina.

**Originalni Whisper Small — WER: 55.00%**

Uklaninjem od ređinog broja listova koj grozdova može se poboštiti provetravanje, ali preterano iz lagani i direktnom suncu može povećeti rizi koj od ožegotina.

**Fine-tuned Whisper Small — WER: 30.00%**

Uklanjanje modređenog broja listova ako groznova može se pobošati provetravanje, ali preterano izlaganje i direktnom suncu može povećati rizik od ožegotina.

**Smanjenje WER-a: 25.00 procentnih poena**

---



### Primer 4 — snimak ID 789

**Referentni tekst:**

Integrisana zaštita bilja kombinuje pravilnu agrotehniku, praćenje štetočina, biološke metode i ciljanu primenu hemijskih sredstava samo kada je to potrebno.

**Originalni Whisper Small — WER: 50.00%**

Integrosana zaštita bila kombino je pravilno agrotehniku, praćenje šte točina, biološke metode i ciljenu priminu chemiskih sredstava samo kada je to potrebno.

**Fine-tuned Whisper Small — WER: 25.00%**

Integrosana zaštita bilja kombino je pravilnoj agrotehniku, praćenje štetočina, biološke metode i ciljenu primenu hemijskih sredstava samo kada je to potrebno.

**Smanjenje WER-a: 25.00 procentnih poena**

---



### Primer 5 — snimak ID 780

**Referentni tekst:**

Nakon žetve proizvođač može uporediti prinos različitih delova parcele sa rezultatima analize zemljišta kako bi lakše pronašao moguće uzroke prostornih razlika.

**Originalni Whisper Small — WER: 38.10%**

Nakon žetve proizvođeč može uporediti prinos razičitih delova par cele sa rezultatima na lize zemljišta kako bi lakše pronošao mogiće uzroke prostornih razlika.

**Fine-tuned Whisper Small — WER: 14.29%**

Nako žetve proizvođač može uporediti prinos različitih delova parcele sa rezultatima na lize zemljišta kako bi lakše pronašao moguće uzroke prostornih razlika.

**Smanjenje WER-a: 23.81 procentnih poena**

---


ZAVRŠNI REZULTAT DOMENSKE ADAPTACIJE
Originalni Whisper Small: 31.01% WER
Fine-tuned Whisper Small: 17.32% WER
Relativno smanjenje WER-a: 44.15%


## Zaključak

Domenska adaptacija Whisper Small modela smanjila je korpusni WER sa 31,01% na 17,32%, odnosno za 13,69 procentnih poena (44,15% relativno). Korpusni CER smanjen je sa 5,91% na 3,23%.

Rezultati na izdvojenom test skupu sa novim govornikom pokazuju uspešno prilagođavanje modela govoru iz oblasti poljoprivrede. Generalizacija na druge govorne skupove analizirana je u zasebnom eksperimentu.